Run this notebook in Google Colab after notebooks 2, 3, and 4. Kaggle credentials are loaded from Colab secrets.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import plotly.express as px

try:
    from google.colab import userdata
    key = userdata.get("KAGGLE_API_TOKEN")
    if key:
        os.environ["KAGGLE_API_TOKEN"] = key.strip()
except ImportError:
    pass

In [2]:
# Install Optuna only when this runtime needs it.
import importlib.metadata
import subprocess
import sys

try:
    optuna_version = importlib.metadata.version("optuna")
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    optuna_version = importlib.metadata.version("optuna")
import optuna
print("Optuna:", optuna_version)

Optuna: 5.0.0


In [3]:
# Install the Kaggle CLI only when this Colab runtime needs it.
import importlib.metadata
import subprocess
import sys

try:
    kaggle_version = importlib.metadata.version("kaggle")
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    kaggle_version = importlib.metadata.version("kaggle")
print("Kaggle CLI:", kaggle_version)

Kaggle CLI: 2.0.2


In [4]:
# Pin the XGBoost version used for the Colab feature-selection artifacts.
import importlib.metadata
import subprocess
import sys
try:
    installed_xgb = importlib.metadata.version("xgboost")
except importlib.metadata.PackageNotFoundError:
    installed_xgb = None
if installed_xgb != "3.4.1":
    if "xgboost" in sys.modules:
        raise RuntimeError("Restart the Colab runtime before installing XGBoost 3.4.1")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost==3.4.1"])
import xgboost as xgb
assert xgb.__version__ == "3.4.1"
print("Feature-selection XGBoost:", xgb.__version__)

Feature-selection XGBoost: 3.4.1


In [5]:
SEED = 42

# Helper functions

In [6]:
import warnings
from contextlib import nullcontext

def reduce_memory(df: pd.DataFrame, verbose: bool = True, ignore_warnings: bool = False, inplace: bool = True) -> pd.DataFrame:
    """
    Downcast numeric columns to reduce DataFrame memory usage.
    """
    if not inplace:
        df = df.copy()

    start_mem = df.memory_usage(deep=True).sum() / 1e6

    warning_context = warnings.catch_warnings() if ignore_warnings else nullcontext()

    with warning_context:
        if ignore_warnings:
            warnings.filterwarnings(
                "ignore",
                category=RuntimeWarning,
                message="overflow encountered in cast",
            )

        for col in df.columns:
            series = df[col]

            if pd.api.types.is_integer_dtype(series):
                c_min = series.min()
                c_max = series.max()

                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = series.astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = series.astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = series.astype(np.int32)
                else:
                    df[col] = series.astype(np.int64)

            elif pd.api.types.is_float_dtype(series):
                c_min = series.min()
                c_max = series.max()

                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = series.astype(np.float32)
                else:
                    df[col] = series.astype(np.float64)

    end_mem = df.memory_usage(deep=True).sum() / 1e6

    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem if start_mem else 0
        print(f"Memory: {start_mem:.2f} MB → {end_mem:.2f} MB ({reduction:.1f}% reduction)")

    return df

# Data load

In [7]:
import subprocess

archive = Path("/content/ieee-fraud-detection.zip")
required_csvs = [Path("/content/train_transaction.csv"), Path("/content/train_identity.csv")]
if not all(path.exists() for path in required_csvs):
    if not archive.exists():
        subprocess.run(
            ["kaggle", "competitions", "download", "-c", "ieee-fraud-detection", "-p", "/content"],
            check=True,
        )
    subprocess.run(["unzip", "-qo", str(archive), "-d", "/content"], check=True)
if not all(path.exists() for path in required_csvs):
    raise FileNotFoundError("Kaggle training CSVs are missing from /content")
print("Training CSVs ready in /content")


Training CSVs ready in /content


In [8]:
import json
import hashlib
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = Path("/content/")
TRAIN_TRANSACTION = BASE_PATH / "train_transaction.csv"
TRAIN_IDENTITY = BASE_PATH / "train_identity.csv"
ARTIFACTS_PATH = Path("/content/drive/My Drive/ML/Fraud Detection/Data/Artifacts")
with open(ARTIFACTS_PATH / "selected_v_features.json") as f:
    selected_v_features = json.load(f)
with open(ARTIFACTS_PATH / "selected_id_features.json") as f:
    selected_id_features = json.load(f)
with open(ARTIFACTS_PATH / "v_feature_selection_metadata.json") as f:
    v_selection_metadata = json.load(f)
with open(ARTIFACTS_PATH / "id_feature_selection_metadata.json") as f:
    id_selection_metadata = json.load(f)

# Reject a stale or changed Drive list before loading the wide CSVs.
for name, features, metadata in [
    ("V", selected_v_features, v_selection_metadata),
    ("ID", selected_id_features, id_selection_metadata),
]:
    actual_sha256 = hashlib.sha256(json.dumps(features, separators=(",", ":")).encode()).hexdigest()
    if (metadata.get("feature_sha256") != actual_sha256
            or metadata.get("xgboost_version") != "3.4.1"):
        raise ValueError(f"Stale {name} features in Drive. Rerun notebooks 2 and 3 with XGBoost 3.4.1.")

# Confirm both selectors and this model notebook use the same raw files.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for source_name, path in [("transaction", TRAIN_TRANSACTION), ("identity", TRAIN_IDENTITY)]:
    actual_sha256 = sha256_file(path)
    key = f"{source_name}_csv_sha256"
    if (v_selection_metadata.get(key) != actual_sha256
            or id_selection_metadata.get(key) != actual_sha256):
        raise ValueError(f"{source_name} CSV differs from the files used in notebooks 2 and 3")

# The selected ID list can include derived missing indicators. Read their source columns.
selected_id_source = {name.removesuffix("_is_missing") for name in selected_id_features}


Mounted at /content/drive


In [9]:
import gc

# Parse only columns used by the models. This avoids holding all 339 V columns
# and all identity columns while subsequently selecting a smaller frame.
train_transaction = reduce_memory(pd.read_csv(
    TRAIN_TRANSACTION,
    usecols=lambda col: not col.startswith("V") or col in selected_v_features,
), True)
train_identity = reduce_memory(pd.read_csv(
    TRAIN_IDENTITY,
    usecols=lambda col: not col.startswith("id_") or col in selected_id_source,
), True)
train_df_merged = pd.merge(train_transaction, train_identity, on="TransactionID", how="left")
del train_transaction, train_identity
gc.collect()
print("Loaded", len(train_df_merged), "transactions with selected features")


Memory: 1642.56 MB → 1001.83 MB (39.0% reduction)
Memory: 124.18 MB → 114.95 MB (7.4% reduction)
Loaded 590540 transactions with selected features


# Feature reduction

Rerun notebooks 2 and 3 before this notebook. Their supervised feature selectors fit only on the earlier 80% of transactions and save lists plus selection metadata to the Artifacts folder. This notebook verifies the training-period boundary before loading either list. The final 20% time holdout is reserved for model evaluation.

In [10]:
# Fail fast if the lists were made for a different period or split.
time_order = train_df_merged["TransactionDT"].sort_values(kind="stable").index
split_idx = int(len(time_order) * 0.8)
expected_train_idx, expected_holdout_idx = time_order[:split_idx], time_order[split_idx:]
expected_time_max = float(train_df_merged.loc[expected_train_idx, "TransactionDT"].max())
expected_holdout_min = float(train_df_merged.loc[expected_holdout_idx, "TransactionDT"].min())
for name, metadata, features in [
    ("V", v_selection_metadata, selected_v_features),
    ("ID", id_selection_metadata, selected_id_features),
]:
    if (metadata.get("xgboost_version") != "3.4.1"
            or metadata["n_selection_rows"] != len(expected_train_idx)
            or metadata["selection_time_max"] != expected_time_max
            or metadata["holdout_time_min"] != expected_holdout_min
            or metadata["n_selected_features"] != len(features)
            or metadata["gain_fraction"] != 0.95):
        raise ValueError(f"Stale {name} feature list. Rerun notebooks 2 and 3 in order.")
print("Validated training-period feature lists:", len(selected_v_features), "V and", len(selected_id_features), "ID")


Validated training-period feature lists: 229 V and 28 ID


In [11]:
assert set(col for col in train_df_merged if col.startswith("V")) <= set(selected_v_features)


In [12]:
# Materialize only selected missing indicators, without copying the whole table.
for col in selected_id_features:
    if col.endswith("_is_missing"):
        source_col = col.removesuffix("_is_missing")
        train_df_merged[col] = train_df_merged[source_col].isna().astype("int8")


In [13]:
# An indicator may be selected without its raw ID source column.
unselected_id_sources = [
    col for col in train_df_merged
    if col.startswith("id_") and col not in selected_id_features
]
train_df_merged.drop(columns=unselected_id_sources, inplace=True)
assert set(col for col in train_df_merged if col.startswith("id_")) <= set(selected_id_features)


In [14]:
TARGET = "isFraud"
DROP_COLS = ["TransactionID"]
TIME_COL = "TransactionDT"

#Train/validation split

In [15]:
X = train_df_merged.drop(columns=[TARGET])
y = train_df_merged[TARGET]
split_idx = int(len(X) * 0.8)
time_order = X[TIME_COL].sort_values(kind="stable").index
train_idx, val_idx = time_order[:split_idx], time_order[split_idx:]
X_train, X_val = X.loc[train_idx], X.loc[val_idx]
y_train, y_val = y.loc[train_idx], y.loc[val_idx]
inner_order = X_train[TIME_COL].sort_values(kind="stable").index
print("Train period:", X_train[TIME_COL].min(), "-", X_train[TIME_COL].max())
print("Validation period:", X_val[TIME_COL].min(), "-", X_val[TIME_COL].max())
del X, y, train_df_merged
gc.collect()


Train period: 86400 - 12192842
Validation period: 12192900 - 15811131


0

# Feature grouping validation

In [16]:
id_like_numeric = [
  "card1", "card2", "card3", "card5",
  "addr1", "addr2"
]

known_categorical = [
  "ProductCD",
  "card4", "card6",
  "P_emaildomain", "R_emaildomain",
  "DeviceType", "DeviceInfo"
]


In [17]:
from pandas.api.types import (
    is_object_dtype,
    is_bool_dtype,
    is_numeric_dtype,
    is_string_dtype,
)
from pandas import CategoricalDtype

def is_categorical_like(series):
    return (
        is_object_dtype(series)
        or isinstance(series.dtype, CategoricalDtype)
        or is_bool_dtype(series)
        or is_string_dtype(series)
    )

def get_feature_groups(df, selected_id_features):
  numeric_cols = []
  categorical_cols = []

  for col in df.columns:
    if col in ["TransactionID", "isFraud"]:
      continue

    if col in id_like_numeric or col in known_categorical:
      categorical_cols.append(col)
      continue

    if col in selected_id_features:
      if is_categorical_like(df[col]):
          categorical_cols.append(col)
      else:
          numeric_cols.append(col)
      continue

    if is_categorical_like(df[col]):
      categorical_cols.append(col)
    elif is_numeric_dtype(df[col]):
      numeric_cols.append(col)

  numeric_cols = [c for c in numeric_cols if c != "TransactionDT"]
  return numeric_cols, categorical_cols

In [18]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
  def __init__(self, min_frequency=0.01, missing_value="Missing"):
    self.min_frequency = min_frequency
    self.missing_value = missing_value
    self.frequent_categories_ = {}

  def fit(self, X, y=None):
    X = pd.DataFrame(X)

    for col in X.columns:
        s = X[col].fillna(self.missing_value).astype(str)
        freq = s.value_counts(normalize=True)
        self.frequent_categories_[col] = set(freq[freq >= self.min_frequency].index)

    return self

  def transform(self, X):
    X = pd.DataFrame(X)

    for col in X.columns:
        s = X[col].fillna(self.missing_value).astype(str)
        keep = self.frequent_categories_[col]
        X[col] = s.where(s.isin(keep), "Rare")

    return X

In [19]:
def add_features(df, use_log: bool = True):
  # drop already returns a new frame; avoid copying it a second time.
  df = df.drop(columns=DROP_COLS, errors="ignore")
  if "TransactionDT" in df.columns:
      df["Transaction_day"] = df["TransactionDT"] // (60 * 60 * 24)
      df["Transaction_hour"] = (df["TransactionDT"] // (60 * 60)) % 24

  if "TransactionAmt" in df.columns and use_log:
      df["TransactionAmt_log1p"] = np.log1p(df["TransactionAmt"])

  return df


In [20]:
X_train_fe = add_features(X_train, use_log=True)
X_val_fe = add_features(X_val, use_log=True)
del X_train, X_val
gc.collect()


0

In [21]:
numeric_cols, categorical_cols = get_feature_groups(X_train_fe, selected_id_features)

In [22]:
missing_selected_id = sorted(set(selected_id_features) - set(X_train_fe.columns))
if missing_selected_id:
    raise ValueError(f"Selected ID features missing from modeling data: {missing_selected_id}")
missing_assignment = [
    col for col in selected_id_features
    if col not in numeric_cols and col not in categorical_cols
]
if missing_assignment:
    raise ValueError(f"Selected ID features lack preprocessing: {missing_assignment}")
print("Preprocessing columns:", len(numeric_cols), "numeric,", len(categorical_cols), "categorical")


Preprocessing columns: 280 numeric, 34 categorical


# Saved model from notebook 4

The saved model must use the same regenerated training-period feature lists. The next cell checks the metadata and stops if the model is stale.

In [23]:
import joblib
import json
from pathlib import Path

MODEL_ARTIFACTS_PATH = Path(ARTIFACTS_PATH) / "models"
MODEL_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

model_path = MODEL_ARTIFACTS_PATH / "final_fraud_model.joblib"
metadata_path = MODEL_ARTIFACTS_PATH / "final_fraud_model_metadata.json"

assert model_path.exists(), f"Model file not found: {model_path}"
assert metadata_path.exists(), f"Metadata file not found: {metadata_path}"

In [24]:
with open(metadata_path) as f:
    model_metadata = json.load(f)
if (model_metadata.get("selected_v_features") != selected_v_features
        or model_metadata.get("selected_id_features") != selected_id_features
        or model_metadata.get("v_selection_metadata") != v_selection_metadata
        or model_metadata.get("id_selection_metadata") != id_selection_metadata):
    raise ValueError("Saved model uses stale feature lists. Rerun notebook 4 first.")
saved_model = joblib.load(model_path)
selected_threshold = model_metadata["selected_threshold"]
saved_model


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'))]),
                                                  ['TransactionAmt', 'dist1',
                                                   'dist2', 'C1', 'C2', 'C3',
                                                   'C4', 'C5', 'C6', 'C7', 'C8',
                                                   'C9', 'C10', 'C11', 'C12',
                                                   'C13', 'C14', 'D1', 'D2',
                                                   'D3', 'D4', 'D5', 'D6', 'D7',
                                                   'D8', 'D9', 'D10', 'D11',
                                                   'D12', 'D13', ...]),
                                                 ('cat',
                                                  Pipeline...
                                                   'card5', 'card6', 'addr1',
                                                   'addr2', 'P_emaildomain',
                                                   'R_emaildomain', 'M1', 'M2',
                                                   'M3', 'M4', 'M5', 'M6', 'M7',
                                                   'M8', 'M9', 'id_12', 'id_15',
                                                   'id_16', 'id_28', 'id_29',
                                                   'id_30', 'id_31', 'id_33',
                                                   'id_34', 'id_35', ...])])),
                ('model',
                 LGBMClassifier(class_weight='balanced', learning_rate=0.05,
                                metric='average_precision', n_estimators=1000,
                                n_jobs=-1, objective='binary',
                                random_state=42))])

# Optuna on an inner time split

Tune only inside the earlier 80% training period. The final 20% time holdout is evaluated after hyperparameters are chosen. The feature lists were ranked on the whole earlier 80%, including the inner validation slice, so the inner Optuna score can be optimistic. A fully independent inner score would require refitting feature selection inside each tuning split. Repeatedly adjusting the search after viewing the final holdout score would also make that holdout exploratory.
The tuned model is saved as a candidate with its holdout score; notebook 4's selected model remains the primary artifact unless the comparison supports a change.


In [25]:
from lightgbm import LGBMClassifier
from sklearn.preprocessing import OrdinalEncoder

# Numeric pipeline (same as baseline)
numeric_pipeline_tree = Pipeline(steps=[
  ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
  # ("scaler", StandardScaler()) - No Need for scaling in tree models
])

# Categorical pipeline for tree models (using OrdinalEncoder)
categorical_pipeline_tree = Pipeline(steps=[
  ("rare", RareCategoryGrouper(min_frequency=0.005)),
  ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
  ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

# Preprocessor for the tree model
preprocessor_tree = ColumnTransformer(
  transformers=[
    ("num", numeric_pipeline_tree, numeric_cols),
    ("cat", categorical_pipeline_tree, categorical_cols)
  ],
  remainder="drop"
)

In [26]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score

# Inner split: first 64% of all transactions for fitting, next 16% for tuning.
inner_cutoff = int(len(X_train_fe) * 0.8)
inner_train_idx = inner_order[:inner_cutoff]
inner_val_idx = inner_order[inner_cutoff:]
X_inner_train, X_inner_val = X_train_fe.loc[inner_train_idx], X_train_fe.loc[inner_val_idx]
y_inner_train, y_inner_val = y_train.loc[inner_train_idx], y_train.loc[inner_val_idx]

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 64),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": SEED,
        "n_jobs": -1,
        "class_weight": "balanced",
        "verbose": -1,
    }

    model = Pipeline(steps=[
        ("preprocess", preprocessor_tree),
        ("model", LGBMClassifier(**params))
    ])

    model.fit(X_inner_train, y_inner_train)

    y_val_proba = model.predict_proba(X_inner_val)[:, 1]

    return average_precision_score(y_inner_val, y_val_proba)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=10)

study.best_value, study.best_params



[I 2026-11-12 11:36:06,717] A new study created in memory with name: no-name-dc2b517f-641b-41ba-9f33-165bf822ab72
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-11-12 11:41:10,278] Trial 0 finished with value: 0.6015057537336954 and parameters: {'learning_rate': 0.02757359293934948, 'num_leaves': 62, 'max_depth': 8, 'min_child_samples': 128, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 0.6015057537336954.
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-11-12 11:44:25,555] Trial 1 finished with value: 0.5663651229786718 and parameters: {'learning_rate': 0.05092911283

(0.6015057537336954,
 {'learning_rate': 0.02757359293934948,
  'num_leaves': 62,
  'max_depth': 8,
  'min_child_samples': 128,
  'subsample': 0.6624074561769746,
  'colsample_bytree': 0.662397808134481,
  'reg_alpha': 3.3323645788192616e-08,
  'reg_lambda': 0.6245760287469893})

In [27]:
best_params = {
      "objective": "binary",
      "metric": "average_precision",
      "n_estimators": 1000,
      "random_state": SEED,
      "n_jobs": -1,
      "class_weight": "balanced",
      "subsample_freq": 1,
      "verbose": -1,
      **study.best_params,
  }

best_lgbm_model = Pipeline(steps=[
    ("preprocess", preprocessor_tree),
    ("model", LGBMClassifier(**best_params))
])

best_lgbm_model.fit(X_train_fe, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'))]),
                                                  ['TransactionAmt', 'dist1',
                                                   'dist2', 'C1', 'C2', 'C3',
                                                   'C4', 'C5', 'C6', 'C7', 'C8',
                                                   'C9', 'C10', 'C11', 'C12',
                                                   'C13', 'C14', 'D1', 'D2',
                                                   'D3', 'D4', 'D5', 'D6', 'D7',
                                                   'D8', 'D9', 'D10', 'D11',
                                                   'D12', 'D13', ...]),
                                                 ('cat',
                                                  Pipeline...
                 LGBMClassifier(class_weight='balanced',
                                colsample_bytree=0.662397808134481,
                                learning_rate=0.02757359293934948, max_depth=8,
                                metric='average_precision',
                                min_child_samples=128, n_estimators=1000,
                                n_jobs=-1, num_leaves=62, objective='binary',
                                random_state=42,
                                reg_alpha=3.3323645788192616e-08,
                                reg_lambda=0.6245760287469893,
                                subsample=0.6624074561769746, subsample_freq=1,
                                verbose=-1))])

In [28]:
from sklearn.metrics import average_precision_score, roc_auc_score

# Evaluate best_lgbm_model
y_val_proba_best_lgbm = best_lgbm_model.predict_proba(X_val_fe)[:, 1]
pr_auc_best_lgbm = average_precision_score(y_val, y_val_proba_best_lgbm)
roc_auc_best_lgbm = roc_auc_score(y_val, y_val_proba_best_lgbm)

# Evaluate saved_model
y_val_proba_saved = saved_model.predict_proba(X_val_fe)[:, 1]
pr_auc_saved = average_precision_score(y_val, y_val_proba_saved)
roc_auc_saved = roc_auc_score(y_val, y_val_proba_saved)

comparison_df = pd.DataFrame({
    'Metric': ['PR-AUC', 'ROC-AUC'],
    'Best LGBM Model': [pr_auc_best_lgbm, roc_auc_best_lgbm],
    'Saved Model': [pr_auc_saved, roc_auc_saved]
})

print("Model Comparison on Validation Set:")
print(comparison_df.to_markdown(index=False))


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model Comparison on Validation Set:
| Metric   |   Best LGBM Model |   Saved Model |
|:---------|------------------:|--------------:|
| PR-AUC   |          0.539675 |      0.535803 |
| ROC-AUC  |          0.914412 |      0.904712 |


In [29]:
candidate_model_path = MODEL_ARTIFACTS_PATH / "tuned_lightgbm_candidate.joblib"
candidate_metadata_path = MODEL_ARTIFACTS_PATH / "tuned_lightgbm_candidate_metadata.json"

# Save the tuned candidate
joblib.dump(best_lgbm_model, candidate_model_path)
print(f"Tuned candidate saved to: {candidate_model_path}")

# Record the tuning and holdout comparison
candidate_model_metadata = {
    "model_name": "Tuned LightGBM candidate",
    "beats_saved_model_on_holdout": bool(pr_auc_best_lgbm > pr_auc_saved),
    "saved_model_holdout_pr_auc": float(pr_auc_saved),
    "selection_metric": "pr_auc",
    "seed": SEED,
    "best_params": study.best_params,
    "inner_validation_pr_auc": study.best_value,
    "holdout_pr_auc": float(pr_auc_best_lgbm),
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "selected_v_features": selected_v_features,
    "selected_id_features": selected_id_features,
}

# Save the tuned candidate's metadata
with open(candidate_metadata_path, 'w') as f:
    json.dump(candidate_model_metadata, f, indent=2)
print(f"Tuned candidate metadata saved to: {candidate_metadata_path}")

Tuned candidate saved to: /content/drive/My Drive/ML/Fraud Detection/Data/Artifacts/models/tuned_lightgbm_candidate.joblib
Tuned candidate metadata saved to: /content/drive/My Drive/ML/Fraud Detection/Data/Artifacts/models/tuned_lightgbm_candidate_metadata.json
